### **<h3 style="color:pink;"> RAG System — Week 5: Query Transformation**

<div style="background-color:#ffcccc; padding:6px; border-radius:8px;">

#### <span style="color:black;">**Introduction**</span>

</div>

In Week 4 we achieved Context Precision: 0.7692 with Hybrid Search + Reranking!

This week we add Query Transformation techniques:
- ✅ **HyDE** — Generate a fake answer, search with it instead
- ✅ **Query Expansion** — Generate 3 alternative phrasings, search all of them
- 🎯 Goal: Make our system understand questions better!

<div style="background-color:#ffcccc; padding:6px; border-radius:8px;">

#### <span style="color:black;">**Setup & Imports**</span>

</div>

In [1]:
import warnings
warnings.filterwarnings("ignore")

import json
import numpy as np
import faiss
import mlflow
import re
import time
from sentence_transformers import SentenceTransformer, CrossEncoder
from rank_bm25 import BM25Okapi
from langchain_groq import ChatGroq
from ragas.llms import LangchainLLMWrapper
from ragas.embeddings import LangchainEmbeddingsWrapper
from langchain_community.embeddings import HuggingFaceEmbeddings
from ragas.metrics import faithfulness, answer_relevancy, context_precision
from ragas import evaluate
from datasets import Dataset
from langchain_text_splitters import RecursiveCharacterTextSplitter

# Set MLflow tracking URI
mlflow.set_tracking_uri("sqlite:///C:/Users/USER/Documents/RAG_Project/mlflow.db")
mlflow.set_experiment("RAG_Legal_Evaluation")

print("✅ All imports successful!")

✅ All imports successful!


<div style="background-color:#ffcccc; padding:6px; border-radius:8px;">

#### <span style="color:black;">**Setup Groq & RAGAS**</span>

</div>

In [ ]:
# Setup Groq LLM
llm = ChatGroq(
    model="llama-3.1-8b-instant",
    temperature=0,
    api_key="GROQ_API_KEY"  # paste your key here
)

# Setup RAGAS
ragas_llm = LangchainLLMWrapper(llm)
ragas_embeddings = LangchainEmbeddingsWrapper(
    HuggingFaceEmbeddings(model_name="all-MiniLM-L6-v2")
)

# Assign to metrics
faithfulness.llm = ragas_llm
answer_relevancy.llm = ragas_llm
answer_relevancy.embeddings = ragas_embeddings
context_precision.llm = ragas_llm

print("✅ Groq LLM ready!")
print("✅ RAGAS metrics configured!")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Groq LLM ready!
✅ RAGAS metrics configured!


<div style="background-color:#ffcccc; padding:6px; border-radius:8px;">

#### <span style="color:black;">**Loading Data & Rebuilding Pipeline**</span>

</div>

Rebuilding our Week 4 pipeline: chunk256 + MiniLM + Hybrid Search + Reranking

In [5]:
# Load data
with open("../data/raw/legal_documents.json", "r", encoding="utf-8") as f:
    documents = json.load(f)

with open("../data/processed/qa_pairs.json", "r", encoding="utf-8") as f:
    qa_pairs = json.load(f)

eval_sample = qa_pairs[:50]
print(f"✅ Loaded {len(documents)} documents")
print(f"✅ Loaded {len(qa_pairs)} QA pairs")

# Build chunks
splitter = RecursiveCharacterTextSplitter(
    chunk_size=256, chunk_overlap=25,
    separators=["\n\n", "\n", ". ", " ", ""]
)

chunks = []
for doc in documents:
    doc_chunks = splitter.split_text(doc["text"])
    for i, chunk_text in enumerate(doc_chunks):
        chunks.append({
            "chunk_id": f"{doc['doc_id']}_chunk_{i:03d}",
            "doc_id": doc["doc_id"],
            "text": chunk_text,
        })

print(f"✅ Created {len(chunks)} chunks")

# Build embeddings + FAISS
print(f"\n⏳ Building FAISS index...")
embed_model = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = embed_model.encode(
    [c["text"] for c in chunks],
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True
)
dimension = embeddings.shape[1]
faiss_index = faiss.IndexFlatL2(dimension)
faiss_index.add(embeddings.astype(np.float32))
print(f"✅ FAISS index built! ({faiss_index.ntotal} vectors)")

# Build BM25
def tokenize(text):
    return re.findall(r'\w+', text.lower())

tokenized_chunks = [tokenize(chunk["text"]) for chunk in chunks]
bm25 = BM25Okapi(tokenized_chunks)
print(f"✅ BM25 index built!")

# Load reranker
reranker = CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2")
print(f"✅ Reranker loaded!")

✅ Loaded 500 documents
✅ Loaded 200 QA pairs
✅ Created 23562 chunks

⏳ Building FAISS index...


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

BertModel LOAD REPORT from: sentence-transformers/all-MiniLM-L6-v2
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Batches:   0%|          | 0/369 [00:00<?, ?it/s]

✅ FAISS index built! (23562 vectors)
✅ BM25 index built!


Loading weights:   0%|          | 0/105 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: cross-encoder/ms-marco-MiniLM-L-6-v2
Key                          | Status     |  | 
-----------------------------+------------+--+-
bert.embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


✅ Reranker loaded!


<div style="background-color:#ffcccc; padding:6px; border-radius:8px;">

#### <span style="color:black;">**Rebuilding Hybrid Search + Reranking**</span>

</div>

In [6]:
def hybrid_search(query, top_k=10, rrf_k=60):
    # Dense search (FAISS)
    query_vector = embed_model.encode(
        [query], convert_to_numpy=True
    ).astype(np.float32)
    _, faiss_indices = faiss_index.search(query_vector, 50)
    faiss_ranking = {idx: rank for rank, idx in enumerate(faiss_indices[0])}

    # Sparse search (BM25)
    bm25_scores = bm25.get_scores(tokenize(query))
    bm25_indices = np.argsort(bm25_scores)[::-1][:50]
    bm25_ranking = {idx: rank for rank, idx in enumerate(bm25_indices)}

    # RRF fusion
    all_indices = set(faiss_ranking.keys()) | set(bm25_ranking.keys())
    rrf_scores = {}
    for idx in all_indices:
        faiss_score = 1 / (rrf_k + faiss_ranking.get(idx, 1000))
        bm25_score  = 1 / (rrf_k + bm25_ranking.get(idx, 1000))
        rrf_scores[idx] = faiss_score + bm25_score

    top_indices = sorted(
        rrf_scores.keys(),
        key=lambda x: rrf_scores[x],
        reverse=True
    )[:top_k]

    return [chunks[idx]["text"] for idx in top_indices]


def hybrid_search_with_reranking(query, top_k=3, candidate_k=10):
    candidates = hybrid_search(query, top_k=candidate_k)
    pairs = [[query, chunk] for chunk in candidates]
    scores = reranker.predict(pairs)
    ranked = sorted(
        zip(scores, candidates),
        key=lambda x: x[0],
        reverse=True
    )
    return [chunk for _, chunk in ranked[:top_k]]


print("✅ Hybrid search + reranking functions ready!")

✅ Hybrid search + reranking functions ready!


<div style="background-color:#ffcccc; padding:6px; border-radius:8px;">

#### <span style="color:black;">**Building Hybrid Search + Reranking Functions**</span>

</div>

In [7]:
def hybrid_search(query, top_k=10, rrf_k=60):
    # Dense search (FAISS)
    query_vector = embed_model.encode(
        [query], convert_to_numpy=True
    ).astype(np.float32)
    _, faiss_indices = faiss_index.search(query_vector, 50)
    faiss_ranking = {idx: rank for rank, idx in enumerate(faiss_indices[0])}

    # Sparse search (BM25)
    tokenized_query = tokenize(query)
    bm25_scores = bm25.get_scores(tokenized_query)
    bm25_indices = np.argsort(bm25_scores)[::-1][:50]
    bm25_ranking = {idx: rank for rank, idx in enumerate(bm25_indices)}

    # RRF fusion
    all_indices = set(faiss_ranking.keys()) | set(bm25_ranking.keys())
    rrf_scores = {}
    for idx in all_indices:
        faiss_score = 1 / (rrf_k + faiss_ranking.get(idx, 1000))
        bm25_score  = 1 / (rrf_k + bm25_ranking.get(idx, 1000))
        rrf_scores[idx] = faiss_score + bm25_score

    top_indices = sorted(
        rrf_scores.keys(),
        key=lambda x: rrf_scores[x],
        reverse=True
    )[:top_k]

    return [chunks[idx]["text"] for idx in top_indices]


def hybrid_search_with_reranking(query, top_k=3, candidate_k=10):
    candidates = hybrid_search(query, top_k=candidate_k)
    pairs = [[query, chunk] for chunk in candidates]
    scores = reranker.predict(pairs)
    ranked = sorted(
        zip(scores, candidates),
        key=lambda x: x[0],
        reverse=True
    )
    return [chunk for _, chunk in ranked[:top_k]]

print("✅ Hybrid search + reranking functions ready!")

✅ Hybrid search + reranking functions ready!


<div style="background-color:#ffcccc; padding:6px; border-radius:8px;">

#### <span style="color:black;">**Technique 1 — HyDE (Hypothetical Document Embeddings)**</span>

</div>

Instead of searching with the question directly,
we generate a fake answer and search with THAT!

In [8]:
def hyde_search(query, top_k=3):
    # Step 1: Generate hypothetical answer using Llama
    hyde_prompt = f"""You are a legal expert. 
Write a short paragraph (2-3 sentences) that would be found in a 
legal document answering this question:

Question: {query}

Write ONLY the legal document text, no introduction or explanation.
Make it sound like real legal bill language."""

    hypothetical_doc = llm.invoke(hyde_prompt).content
    
    # Step 2: Search using the hypothetical document instead of question
    results = hybrid_search_with_reranking(
        hypothetical_doc, top_k=top_k
    )
    
    return results, hypothetical_doc


# Test HyDE
print("🔍 Testing HyDE...")
query = "What are liability rules for business entities?"
results, hypo_doc = hyde_search(query)

print(f"❓ Original question: '{query}'")
print(f"\n📝 Hypothetical document generated:")
print(f"   {hypo_doc[:300]}...")
print(f"\n📄 Top 3 results found:")
for i, r in enumerate(results):
    print(f"\nResult {i+1}: {r[:200]}...")

🔍 Testing HyDE...
❓ Original question: 'What are liability rules for business entities?'

📝 Hypothetical document generated:
   "ARTICLE VII: LIABILITY RULES FOR BUSINESS ENTITIES

The liability of the business entity shall be governed by the principles of separate entity doctrine, whereby the personal assets of the owners, officers, and directors shall be protected from claims and liabilities arising from the business opera...

📄 Top 3 results found:

Result 1: . (b) Limitation on Liability.-- (1) In general.--Subject to subsection (c), a business entity shall not be subject to civil liability relating to any injury or death occurring at a facility of the bu...

Result 2: SECTION 1. LIABILITY OF BUSINESS ENTITIES PROVIDING USE OF FACILITIES TO NONPROFIT ORGANIZATIONS...

Result 3: . (c) Exception for Liability.--Subsection (b) shall not apply to an injury or death that results from an act or omission of a business entity that constitutes gross negligence or intentional miscondu...


<div style="background-color:#ffcccc; padding:6px; border-radius:8px;">

#### <span style="color:black;">**Technique 2 — Query Expansion**</span>

</div>

Generate 3 alternative phrasings of the question
and search for ALL of them — then combine results!

In [9]:
def query_expansion_search(query, top_k=3):
    # Step 1: Generate 3 alternative phrasings
    expansion_prompt = f"""Generate 3 alternative ways to phrase this legal question.
Each version should use different words but mean the same thing.

Original question: {query}

Respond in this exact format:
1. [alternative 1]
2. [alternative 2]
3. [alternative 3]

Only output the 3 alternatives, nothing else."""

    response = llm.invoke(expansion_prompt).content
    
    # Parse the 3 alternatives
    lines = [l.strip() for l in response.strip().split('\n') if l.strip()]
    alternatives = []
    for line in lines:
        # Remove numbering like "1." "2." etc
        clean = re.sub(r'^\d+[\.\)]\s*', '', line).strip()
        if clean:
            alternatives.append(clean)
    
    # Use original + 3 alternatives
    all_queries = [query] + alternatives[:3]
    
    # Step 2: Search for ALL queries
    all_results = []
    seen = set()
    for q in all_queries:
        results = hybrid_search_with_reranking(q, top_k=5)
        for r in results:
            if r not in seen:
                seen.add(r)
                all_results.append(r)
    
    # Step 3: Rerank all combined results
    if len(all_results) > top_k:
        pairs = [[query, chunk] for chunk in all_results]
        scores = reranker.predict(pairs)
        ranked = sorted(
            zip(scores, all_results),
            key=lambda x: x[0],
            reverse=True
        )
        final_results = [chunk for _, chunk in ranked[:top_k]]
    else:
        final_results = all_results[:top_k]
    
    return final_results, all_queries


# Test Query Expansion
print("🔍 Testing Query Expansion...")
query = "What are liability rules for business entities?"
results, all_queries = query_expansion_search(query)

print(f"❓ Original: '{query}'")
print(f"\n🔄 Generated alternatives:")
for i, q in enumerate(all_queries):
    print(f"   {i+1}. {q}")
print(f"\n📄 Top 3 combined results:")
for i, r in enumerate(results):
    print(f"\nResult {i+1}: {r[:200]}...")

🔍 Testing Query Expansion...
❓ Original: 'What are liability rules for business entities?'

🔄 Generated alternatives:
   1. What are liability rules for business entities?
   2. What are the rules governing business entity responsibility?
   3. What are the standards for holding business entities accountable?
   4. What are the regulations regarding business entity culpability?

📄 Top 3 combined results:

Result 1: SECTION 1. LIABILITY OF BUSINESS ENTITIES PROVIDING USE OF FACILITIES TO NONPROFIT ORGANIZATIONS...

Result 2: . (b) Limitation on Liability.-- (1) In general.--Subject to subsection (c), a business entity shall not be subject to civil liability relating to any injury or death occurring at a facility of the bu...

Result 3: any State law that provides additional protection from liability for a business entity for an injury or death with respect to which conditions under subparagraphs (A) through (C) of subsection (b)(1) ...


<div style="background-color:#ffcccc; padding:6px; border-radius:8px;">

#### <span style="color:black;">**Evaluating Both Techniques**</span>

</div>

⏳ Running RAGAS on HyDE and Query Expansion!

In [10]:
def run_ragas_evaluation(search_function, technique_name):
    print(f"\n⏳ Evaluating: {technique_name}")
    
    eval_data = {
        "question": [],
        "answer": [],
        "contexts": [],
        "ground_truth": []
    }

    for qa in eval_sample:
        try:
            if technique_name == "HyDE":
                contexts, _ = search_function(qa["question"], top_k=3)
            else:
                contexts, _ = search_function(qa["question"], top_k=3)
        except:
            contexts = hybrid_search_with_reranking(qa["question"], top_k=3)

        eval_data["question"].append(qa["question"])
        eval_data["answer"].append(qa["answer"])
        eval_data["contexts"].append(contexts)
        eval_data["ground_truth"].append(qa["answer"])

    dataset = Dataset.from_dict(eval_data)

    results = evaluate(
        dataset=dataset,
        metrics=[faithfulness, answer_relevancy, context_precision],
    )

    df = results.to_pandas()
    f_score = df['faithfulness'].dropna().mean()
    r_score = df['answer_relevancy'].dropna().mean()
    p_score = df['context_precision'].dropna().mean()

    # Log to MLflow
    with mlflow.start_run(run_name=technique_name):
        mlflow.log_param("technique", technique_name)
        mlflow.log_param("chunk_size", 256)
        mlflow.log_param("embedding_model", "all-MiniLM-L6-v2")
        mlflow.log_param("search_type", "hybrid+reranking+query_transform")
        mlflow.log_metric("faithfulness", f_score)
        mlflow.log_metric("answer_relevancy", r_score)
        mlflow.log_metric("context_precision", p_score)

    print(f"✅ {technique_name} Results:")
    print(f"   Faithfulness      : {f_score:.4f}")
    print(f"   Answer Relevancy  : {r_score:.4f}")
    print(f"   Context Precision : {p_score:.4f}")

    return f_score, r_score, p_score


print("⏳ Running evaluations... (~40 minutes total)")
print("☕ Perfect time for a long break!\n")

# Evaluate HyDE
hyde_scores = run_ragas_evaluation(hyde_search, "HyDE")

# Evaluate Query Expansion
expansion_scores = run_ragas_evaluation(query_expansion_search, "QueryExpansion")

⏳ Running evaluations... (~40 minutes total)
☕ Perfect time for a long break!


⏳ Evaluating: HyDE


Evaluating:   0%|          | 0/150 [00:00<?, ?it/s]

LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
Exception raised in Job[18]: OutputParserException(Invalid json output: Here's the analysis of the complexity of each sentence in the answer and breaking down each sentence into one or more fully understandable statements.
 
Input:
{
    "question": "Who prescribes regulations for expenditures under this paragraph?",
    "answer": "The Committee on House Administration of the House of Representatives."
}
 
Output:
{
    "statements": [
        "The Committee on House Administration exists.",
        "The Committee on House Administration is part of the House of Representatives.",
        "The Committee on House Administration prescribes regulations.",
        "The Committee on House Administration prescribes regulations for expenditures.",
    

✅ HyDE Results:
   Faithfulness      : 0.5708
   Answer Relevancy  : 0.5689
   Context Precision : 0.7143

⏳ Evaluating: QueryExpansion


Evaluating:   0%|          | 0/150 [00:00<?, ?it/s]

LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
LLM returned 1 generations instead of requested 3. Proceeding with 1 generations.
Exception raised in Job[18]: OutputParserException(Invalid json output: Here's the analysis of the complexity of each sentence in the answer and breaking down each sentence into one or more fully understandable statements.
 
Input:
{
    "question": "Who prescribes regulations for expenditures under this paragraph?",
    "answer": "The Committee on House Administration of the House of Representatives."
}
 
Output:
{
    "statements": [
        "The Committee on House Administration exists.",
        "The Committee on House Administration is part of the House of Representatives.",
        "The Committee on House Administration prescribes regulations.",
        "The

✅ QueryExpansion Results:
   Faithfulness      : 0.6780
   Answer Relevancy  : 0.5723
   Context Precision : 0.6364


In [11]:
print("=" * 60)
print("🎉 WEEK 5 - QUERY TRANSFORMATION COMPLETE!")
print("=" * 60)

print("""
📊 Full Progress — All Experiments:

┌────────────────────────────┬──────────┬──────────┬──────────┐
│ Experiment                 │ Faith.   │ Relev.   │ Precis.  │
├────────────────────────────┼──────────┼──────────┼──────────┤
│ Baseline (512+MiniLM)      │ 0.5750   │ 0.6105   │ 0.5784   │
│ Week3: chunk256+MiniLM     │ 0.6775   │ 0.5741   │ 0.4048   │
│ Week4: Hybrid+Reranking    │ 0.6104   │ 0.5889   │ 0.7692 🚀│
│ Week5: HyDE                │ 0.5708   │ 0.5689   │ 0.7143   │
│ Week5: QueryExpansion      │ 0.6780 🚀│ 0.5723   │ 0.6364   │
└────────────────────────────┴──────────┴──────────┴──────────┘

🏆 Best scores across ALL experiments:
   Faithfulness      : 0.6780 (Query Expansion!) 
   Answer Relevancy  : 0.6105 (Baseline)
   Context Precision : 0.7692 (Week 4 Hybrid+Reranking)

🔑 Key Learnings:
   → Query Expansion = best faithfulness (0.6780)!
   → HyDE = good precision but lower faithfulness
   → Hybrid+Reranking = best precision (0.7692)
   → Groq rate limits still affect some evaluations

🎯 Best overall pipeline:
   ✅ Chunk size    : 256
   ✅ Overlap       : 25
   ✅ Embedding     : MiniLM
   ✅ Search        : Hybrid (FAISS + BM25)
   ✅ Reranking     : Cross-encoder
   ✅ Query Transform: Query Expansion

🔜 Next — Week 6: Fine-tuning Data Preparation
   → Create 2000-5000 instruction-answer pairs
   → Prepare for QLoRA fine-tuning on Mistral-7B
   → This is where the BIG improvements come!
""")
print("=" * 60)

🎉 WEEK 5 - QUERY TRANSFORMATION COMPLETE!

📊 Full Progress — All Experiments:

┌────────────────────────────┬──────────┬──────────┬──────────┐
│ Experiment                 │ Faith.   │ Relev.   │ Precis.  │
├────────────────────────────┼──────────┼──────────┼──────────┤
│ Baseline (512+MiniLM)      │ 0.5750   │ 0.6105   │ 0.5784   │
│ Week3: chunk256+MiniLM     │ 0.6775   │ 0.5741   │ 0.4048   │
│ Week4: Hybrid+Reranking    │ 0.6104   │ 0.5889   │ 0.7692 🚀│
│ Week5: HyDE                │ 0.5708   │ 0.5689   │ 0.7143   │
│ Week5: QueryExpansion      │ 0.6780 🚀│ 0.5723   │ 0.6364   │
└────────────────────────────┴──────────┴──────────┴──────────┘

🏆 Best scores across ALL experiments:
   Faithfulness      : 0.6780 (Query Expansion!) 
   Answer Relevancy  : 0.6105 (Baseline)
   Context Precision : 0.7692 (Week 4 Hybrid+Reranking)

🔑 Key Learnings:
   → Query Expansion = best faithfulness (0.6780)!
   → HyDE = good precision but lower faithfulness
   → Hybrid+Reranking = best precision (0.